# Model Implementation: Indirect Prompt Injection Detection

Este notebook entrena y evalúa clasificadores ML clásicos para detectar indirect prompt injection.

**Inputs (de notebook 02):**
- `features/X_linguistic.pkl` — 10 features lingüísticas handcrafted
- `features/df_linguistic.pkl` — DataFrame con nombres de features
- `data/indirect_prompt_injection_bipia_gpt_train.csv` — dataset original (para labels y TF-IDF)

**Estructura:**
1. Carga de Artifactos
2. Feature Sets (Linguistic + TF-IDF)
3. Train/Test Split
4. Entrenamiento y Evaluación de Modelos
5. Comparativa de Resultados (modelos)
6. Analisis de Errores
7. Mejor modelo o fine tuning (preguntar)

In [2]:
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, issparse

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports OK")

Imports OK


---
## Seccion 1 - Carga de Artifactos

In [3]:
def resolve_path(candidates):
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    raise FileNotFoundError(f"None of {candidates} found")

DATA_PATH = resolve_path([
    "../data/indirect_prompt_injection_bipia_gpt_train.csv",
    "data/indirect_prompt_injection_bipia_gpt_train.csv",
])
FEATURES_DIR = resolve_path(["../features", "features"])
MODELS_DIR = Path(FEATURES_DIR.parent / "models")
MODELS_DIR.mkdir(exist_ok=True)

print(f"Data:     {DATA_PATH}")
print(f"Features: {FEATURES_DIR}")
print(f"Models:   {MODELS_DIR}")

Data:     ..\data\indirect_prompt_injection_bipia_gpt_train.csv
Features: ..\features
Models:   ..\models


In [4]:
# Cargar dataset original (para labels y texto)
df = pd.read_csv(DATA_PATH)
y = df["label"].values

print(f"Dataset shape: {df.shape}")
print(f"Labels: {np.bincount(y)} (0=benign, 1=malicious)")

# Cargar features lingüísticas del notebook 02
X_linguistic = joblib.load(FEATURES_DIR / "X_linguistic.pkl")
df_linguistic = joblib.load(FEATURES_DIR / "df_linguistic.pkl")

LINGUISTIC_FEATURES = list(df_linguistic.columns)
print(f"\nX_linguistic shape: {X_linguistic.shape}")
print(f"Feature names: {LINGUISTIC_FEATURES}")

Dataset shape: (70000, 4)
Labels: [35000 35000] (0=benign, 1=malicious)

X_linguistic shape: (70000, 10)
Feature names: ['char_count', 'word_count', 'avg_word_length', 'sentence_count', 'uppercase_ratio', 'special_char_ratio', 'injection_keyword_count', 'question_mark_count', 'imperative_verb_ratio', 'language_switch_flag']


---
## Seccion 2 - Feature sets

Construimos dos feature sets adicionales:
- **TF-IDF sobre `context`**: captura patrones léxicos sin necesitar GPU
- **Combinado**: linguistic + TF-IDF

In [ ]:
# TF-IDF sobre la columna 'context'
tfidf = TfidfVectorizer(
    max_features=5000,
    sublinear_tf=True,       # log(1+tf) — reduce dominancia de términos frecuentes
    ngram_range=(1, 2),      # unigramas + bigramas
    min_df=5,                # ignorar términos muy raros
    strip_accents="unicode",
    analyzer="word",
)

texts = df["context"].astype(str).tolist()
X_tfidf = tfidf.fit_transform(texts)

print(f"X_tfidf shape: {X_tfidf.shape}")
print(f"Sparse matrix density: {X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]):.4f}")

X_tfidf shape: (70000, 5000)
Sparse matrix density: 0.0223


In [6]:
from scipy.sparse import csr_matrix

# Feature matrix combinada: linguistic (dense) + TF-IDF (sparse)
X_linguistic_sparse = csr_matrix(X_linguistic.astype(np.float32))
X_combined = hstack([X_linguistic_sparse, X_tfidf])

print("=== Feature Matrices ===")
print(f"X_linguistic : {X_linguistic.shape}  (dense)")
print(f"X_tfidf      : {X_tfidf.shape}  (sparse)")
print(f"X_combined   : {X_combined.shape}  (sparse)")

=== Feature Matrices ===
X_linguistic : (70000, 10)  (dense)
X_tfidf      : (70000, 5000)  (sparse)
X_combined   : (70000, 5010)  (sparse)


---
## Seccion 3 - Train/Test Split

Split estratificado 80/20 para mantener el balance de clases en ambos sets.

In [7]:
TEST_SIZE = 0.20

# Indices para splits consistentes entre feature sets
idx_train, idx_test = train_test_split(
    np.arange(len(y)),
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

y_train, y_test = y[idx_train], y[idx_test]

# Splits por feature set
X_ling_train, X_ling_test = X_linguistic[idx_train], X_linguistic[idx_test]
X_tfidf_train, X_tfidf_test = X_tfidf[idx_train], X_tfidf[idx_test]
X_comb_train, X_comb_test = X_combined[idx_train], X_combined[idx_test]

print(f"Train size: {len(y_train):,} | Test size: {len(y_test):,}")
print(f"Train class dist: {np.bincount(y_train)}")
print(f"Test  class dist: {np.bincount(y_test)}")

Train size: 56,000 | Test size: 14,000
Train class dist: [28000 28000]
Test  class dist: [7000 7000]


---
## Seccion 4 - Entrenamiento y Evaluación de Modelos


| Modelo | Feature Set | Justificación |
|--------|-------------|---------------|
| Logistic Regression | Combined | Baseline rápido, interpretable |
| Random Forest | Linguistic | Robusto con pocas features, importancias |
| Gradient Boosting | Combined | Mejor rendimiento esperado |

**Métricas:** F1, Precision, Recall, ROC-AUC con dataset balanceado.

In [8]:
def evaluate_model(model, X_test, y_test, model_name: str):
    """Evalúa un modelo ya entrenado y retorna métricas."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    report = classification_report(y_test, y_pred, target_names=["Benign", "Malicious"], output_dict=True)
    auc = roc_auc_score(y_test, y_prob)

    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=["Benign", "Malicious"]))
    print(f"ROC-AUC: {auc:.4f}")

    return {
        "model_name": model_name,
        "accuracy": report["accuracy"],
        "precision": report["Malicious"]["precision"],
        "recall": report["Malicious"]["recall"],
        "f1": report["Malicious"]["f1-score"],
        "roc_auc": auc,
        "y_prob": y_prob,
        "y_pred": y_pred,
    }

### 4A. Logistic Regression (Baseline)

In [ ]:
# Pipeline: StandardScaler (solo linguistic) + LR
# Para X_combined usamos solo el scaler en la parte densa; TF-IDF ya está normalizado
lr_pipeline = Pipeline([
    ("lr", LogisticRegression(
        C=1.0,
        max_iter=1000,
        solver="saga",      # soporta sparse + L1/L2
        penalty="l2",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ))
])

lr_pipeline.fit(X_comb_train, y_train)
results_lr = evaluate_model(lr_pipeline, X_comb_test, y_test, "Logistic Regression (Combined)")

Training Logistic Regression (Combined features)...

  Logistic Regression (Combined)
              precision    recall  f1-score   support

      Benign       0.64      0.55      0.59      7000
   Malicious       0.61      0.70      0.65      7000

    accuracy                           0.62     14000
   macro avg       0.63      0.62      0.62     14000
weighted avg       0.63      0.62      0.62     14000

ROC-AUC: 0.6909


### 4B. Random Forest

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print("Training Random Forest (Linguistic features)...")
rf_model.fit(X_ling_train, y_train)
results_rf = evaluate_model(rf_model, X_ling_test, y_test, "Random Forest (Linguistic)")

Training Random Forest (Linguistic features)...

  Random Forest (Linguistic)
              precision    recall  f1-score   support

      Benign       0.92      0.86      0.89      7000
   Malicious       0.87      0.92      0.89      7000

    accuracy                           0.89     14000
   macro avg       0.89      0.89      0.89     14000
weighted avg       0.89      0.89      0.89     14000

ROC-AUC: 0.9561


### 4C. Gradient Boosting

In [11]:
# Gradient Boosting es más lento en sparse matrices grandes
# Usamos X_linguistic (dense, pequeña) para GBM base
gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    random_state=RANDOM_STATE,
)

print("Training Gradient Boosting (Linguistic features)...")
gb_model.fit(X_ling_train, y_train)
results_gb = evaluate_model(gb_model, X_ling_test, y_test, "Gradient Boosting (Linguistic)")

Training Gradient Boosting (Linguistic features)...

  Gradient Boosting (Linguistic)
              precision    recall  f1-score   support

      Benign       0.84      0.77      0.80      7000
   Malicious       0.79      0.86      0.82      7000

    accuracy                           0.81     14000
   macro avg       0.81      0.81      0.81     14000
weighted avg       0.81      0.81      0.81     14000

ROC-AUC: 0.8934


### 4D. Random Forest + TF-IDF

In [12]:
# RF puede manejar sparse matrices
rf_combined = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print("Training Random Forest (Combined features)...")
rf_combined.fit(X_comb_train, y_train)
results_rf_comb = evaluate_model(rf_combined, X_comb_test, y_test, "Random Forest (Combined)")

Training Random Forest (Combined features)...

  Random Forest (Combined)
              precision    recall  f1-score   support

      Benign       0.99      0.96      0.97      7000
   Malicious       0.96      0.99      0.97      7000

    accuracy                           0.97     14000
   macro avg       0.97      0.97      0.97     14000
weighted avg       0.97      0.97      0.97     14000

ROC-AUC: 0.9953


Notes: k-folds + test, valid, train

Cross-validation, + data check con RF + TF - IDF (n_X)

Puede ser, TDF idf + cuantis (puede ser) 

